# Dataset Audit

This notebook performs the Phase 1 dataset audit for the SEO performance project using the real UrbanScape dataset. The goal is to understand the data before selecting modeling tasks or writing the broader pipeline.

Key questions:
- What is the dataset shape?
- Which variables exist and what types are they?
- Are there missing values, duplicates, or invalid observations?
- What is the temporal range?
- Which features are likely to be valid prediction inputs?
- Which features may represent leakage or future information?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

raw_path = Path('..') / 'data' / 'raw' / 'UrbanScape_Apparel_SEO_Performance_Final_Dataset.xlsx'
raw_path.exists()

In [ ]:
df = pd.read_excel(raw_path)
df.head()

## 1. Dataset structure

We begin with the size and basic schema of the dataset.

In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Columns and dtypes:')
print(df.dtypes.to_string())

## 2. Missing values and duplicates

We inspect whether the dataset is clean enough for reliable modeling or whether it needs cleaning before ML tasks are defined.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print('Missing values:')
print(missing[missing > 0].to_string())
print('\nDuplicate rows:', int(df.duplicated().sum()))

## 3. Categorical and unique-value audit

This helps us understand cardinality, data quality, and whether specific variables are useful as categorical signals or should be bucketed.

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f'\n{col}: unique={df[col].nunique()}')
    print(df[col].dropna().unique()[:10])

## 4. Temporal coverage

The data contains a date field, so we need to determine whether it supports temporal modeling or whether the structure is mainly cross-sectional.

In [ ]:
if 'Date' in df.columns:
    s = pd.to_datetime(df['Date'], errors='coerce')
    print('Date min:', s.min())
    print('Date max:', s.max())
    print('Null dates:', int(s.isna().sum()))
    print('Date span (days):', (s.max() - s.min()).days)

## 5. Numeric summary

We inspect the behavior of the core SEO metrics to understand the scale and distribution of performance signals.

In [ ]:
numeric_cols = [
    'Organic_Traffic',
    'Clicks',
    'Impressions',
    'CTR (%)',
    'Average_Position',
    'Backlinks',
    'Domain_Authority',
    'Bounce_Rate (%)',
    'Pages_per_Session',
    'Conversion_Rate (%)',
    'Organic_Revenue ($)',
    'Page_Load_Time (sec)',
    'Core_Web_Vitals_LCP (sec)',
    'Core_Web_Vitals_FID (ms)',
    'Core_Web_Vitals_CLS',
    'Technical_SEO_Errors',
]

summary = df[numeric_cols].describe().T
summary

## 6. Feature taxonomy

The next step is to classify features into conceptual groups and decide which categories are suitable for modeling.

In [ ]:
feature_groups = {
    'Temporal': ['Date', 'Month', 'Year', 'Quarter', 'Time Of Day'],
    'Search / SEO': ['Primary Keywords', 'Secondary Keywords', 'Long-Tail Keywords', 'Keywords_Ranking', 'Clicks', 'Impressions', 'CTR (%)', 'Average_Position', 'Backlinks', 'Domain_Authority', 'Keyword_Difficulty', 'Indexed_Pages'],
    'Traffic': ['Organic_Traffic', 'Referral_Traffic', 'Organic_vs_Paid_Traffic_Split (%)', 'Mobile_vs_Desktop_Traffic_Split (%)'],
    'Engagement': ['Bounce_Rate (%)', 'Pages_per_Session', 'Average_Session_Duration (sec)', 'New_vs_Returning_Visitors (%)', 'Exit_Rate (%)'],
    'Conversion / Business': ['Conversion_Rate (%)', 'Goal_Completions', 'Organic_Revenue ($)'],
    'Technical SEO / Performance': ['Page_Load_Time (sec)', 'Core_Web_Vitals_LCP (sec)', 'Core_Web_Vitals_FID (ms)', 'Core_Web_Vitals_CLS', 'Technical_SEO_Errors', 'Time_to_First_Byte_TTFB (ms)'],
    'Context / categorical': ['Location', 'Social Media Source', 'Media Type', 'Device Type', 'Top_Landing_Pages'],
}

for group, cols in feature_groups.items():
    available = [c for c in cols if c in df.columns]
    print(f'\n{group}: {available}')

## 7. Potential leakage and prediction-time concerns

The purpose of this audit is not to remove features arbitrarily. It is to identify variables that may be target-derived, mathematically redundant, or unavailable at prediction time.

In [ ]:
leakage_candidates = [
    'CTR (%)',
    'Clicks',
    'Impressions',
    'Average_Position',
    'Conversion_Rate (%)',
    'Organic_Revenue ($)',
    'Goal_Completions',
]

for col in leakage_candidates:
    if col in df.columns:
        print(f'{col}: min={df[col].min()} median={df[col].median()} max={df[col].max()}')

## 8. Recommended next steps

From this audit, the most important next actions are:

1. Confirm the valid target tasks for the dataset.
2. Decide whether the data supports repeated-entity time series modeling.
3. Identify genuine leakage risks before defining the final model tasks.
4. Only then move to data validation, feature engineering, and ML experimentation.

This notebook intentionally stops at audit-stage findings. It should feed the next notebook or project planning decisions without building the full pipeline yet.